In [5]:
import os
import re
import shutil
import filecmp
from datetime import datetime, timezone

# --- CONFIGURATION ---
# Path to your existing mod (the one you want to update, containing your changes)
MOD_SOURCE_DIR = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes"

# Path to the OLD VANILLA game files that your mod was ORIGINALLY based on.
OLD_VANILLA_DIR_REFERENCE = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver"

# Path to the NEWEST vanilla game files (the version you are updating TO)
GAME_VANILLA_DIR_NEW = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game"

# Name of the subfolder to create in the script's execution directory for the output
OUTPUT_SUBFOLDER_NAME = "updated_mod_v7_3way_interactive"

# Folders within the mod/game to process
FOLDERS_TO_PROCESS = ["common", "events"]
# --- END CONFIGURATION ---

# --- Global Variables ---
SCRIPT_EXECUTION_DIR = os.getcwd()
MOD_OUTPUT_DIR = os.path.join(SCRIPT_EXECUTION_DIR, OUTPUT_SUBFOLDER_NAME)
SKIP_ALL_PARAMETER_CONFIRMATIONS = False # For interactive parameter change confirmation

# --- Helper Functions ---
def log_message(message, level="INFO", indent=0):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    indent_space = "  " * indent
    level_str = f"[{level.upper():<7}]"
    print(f"{timestamp} {level_str} {indent_space}{message}")

def get_user_confirmation_for_parameter(prompt_message):
    global SKIP_ALL_PARAMETER_CONFIRMATIONS
    if SKIP_ALL_PARAMETER_CONFIRMATIONS:
        return "yes"

    while True:
        # Shortened prompt for better readability in loop
        response = input(f"[ACTION ] {prompt_message} (y/n/s[kip all this file]/a[pply all this file]): ").strip().lower()
        if response in ["yes", "y"]: return "yes"
        if response in ["no", "n"]: return "no"
        if response in ["skip all", "s", "skipall"]: # Skip all confirmations for *any* file
            log_message("Skipping ALL further parameter confirmations for this ENTIRE script run.", "INFO_DETAIL")
            SKIP_ALL_PARAMETER_CONFIRMATIONS = True
            return "yes" # Treat as yes for the current one
        if response in ["apply all", "a", "applyall"]: # Apply all remaining changes for *this current file only*
             log_message("Applying all remaining parameter changes for the current file without further prompts.", "INFO_DETAIL")
             return "apply_all_for_file"
        print("[ERROR  ] Invalid input. Please enter 'y', 'n', 's', or 'a'.")


def get_file_lines(filepath, context="reading file"):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f: return f.readlines()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f: return f.readlines()
        except Exception as e_inner:
            log_message(f"Could not read '{filepath}' ({context}): {e_inner}", "ERROR", 1)
            return None
    except FileNotFoundError:
        log_message(f"File not found for {context}: '{filepath}'", "WARN ", 1)
        return None
    except Exception as e:
        log_message(f"Generic error {context} '{filepath}': {e}", "ERROR", 1)
        return None

def write_file_lines(filepath, lines):
    try:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, 'w', encoding='utf-8-sig') as f: f.writelines(lines)
        return True
    except Exception as e:
        log_message(f"Error writing to '{filepath}': {e}", "ERROR", 1)
        return False

def parse_parameter_line(line_text):
    """
    Parses 'key = value' and determines if value is "simple".
    Returns (key, value_string, is_simple_value_flag, comment_at_end_string).
    Returns (None, None, False, "") if not a parseable parameter line.
    """
    stripped_line = line_text.strip()
    # Default returns for non-parameter lines or lines to ignore
    if not stripped_line or stripped_line.startswith('#'):
        return None, None, False, ""

    line_no_comment_parts = stripped_line.split('#', 1)
    effective_line = line_no_comment_parts[0].strip()
    comment_at_end = f" #{line_no_comment_parts[1].strip()}" if len(line_no_comment_parts) > 1 else ""

    match = re.match(r'^\s*([\w.-]+)\s*=\s*(.+)$', effective_line)
    if match:
        key = match.group(1)
        value_str = match.group(2).strip()
        
        is_simple = True # Assume simple unless proven otherwise by its content
        
        # Heuristics to determine if the value_str represents a block itself
        # rather than a simple scalar value.
        if value_str == "{": # Value is just an opening brace (block continues on next line)
            is_simple = False
        elif value_str.startswith("{") and value_str.endswith("}"): # Value is like "{ ... }"
            if value_str == "{}": # An empty block {} is simple enough
                is_simple = True
            # If it contains internal structure (further braces or assignments), it's not a simple value.
            elif '{' in value_str[1:-1] or '}' in value_str[1:-1] or '=' in value_str[1:-1]:
                 is_simple = False
            # else: could be something like { 1.0 0.5 0.2 } which is a simple list of values
        elif value_str.endswith("{"): # This indicates the line is starting a block that continues
             is_simple = False
        # Add more conditions if needed to refine "simple" e.g. for lists like `value = { val1 val2 }`

        return key, value_str, is_simple, comment_at_end
    else: # Not a key = value line after stripping comments
        return None, None, False, ""


def find_entry_block_indices(entry_name, content_lines, start_search_from_idx=0):
    entry_start_regex = re.compile(r"^\s*" + re.escape(entry_name) + r"\s*=\s*\{")
    block_start_idx, brace_level = -1, 0
    for i in range(start_search_from_idx, len(content_lines)):
        line, stripped_line = content_lines[i], content_lines[i].strip()
        if block_start_idx == -1:
            if entry_start_regex.match(stripped_line):
                block_start_idx = i
                brace_level = line.count('{') - line.count('}')
                if brace_level <= 0 and '{' in line:
                    if stripped_line.endswith("}") and line.count('{') == line.count('}'): return block_start_idx, i
                    block_start_idx = -1; continue
                elif brace_level == 0 and '{' not in line: block_start_idx = -1; continue
        elif block_start_idx != -1:
            brace_level += line.count('{') - line.count('}')
            if brace_level <= 0: return block_start_idx, i
    return None, None

def get_parameters_within_block(content_lines, block_start_idx, block_end_idx):
    params = {} # key: (value_str, original_comment_str)
    if block_start_idx is None or block_start_idx >= block_end_idx: return params
    for i in range(block_start_idx + 1, block_end_idx): # Iterate lines *inside* the main block braces
        param_key, param_value, is_simple, comment = parse_parameter_line(content_lines[i])
        if param_key and is_simple: # Only store simple parameters
            params[param_key] = (param_value, comment)
    return params

# --- Phase 1: Identify YOUR Specific Tweaks (Mod vs OLD VANILLA) ---
def identify_user_tweaks():
    log_message("Phase 1: Identifying your specific tweaks (Mod vs OLD VANILLA)...", "PHASE")
    # Structure: {'rel/path/file.txt': {
    #                'entry_A': {'param1': ('mod_val', 'old_van_val_or_flag', 'mod_comment')},
    #                'entry_B__ADDED_ENTRY__': "block_content_string_from_mod"
    #             }}
    # Flag for old_van_val: "__PARAM_ADDED_BY_YOU__" if param didn't exist in old vanilla entry
    # Flag for old_van_val: "__ENTRY_WAS_VANILLA_BUT_PARAMS_ALL_NEW__" special case
    all_user_tweaks = {}
    entry_regex = re.compile(r"^\s*([\w.-]+)\s*=\s*\{") # For top-level entries
    files_with_identified_tweaks = 0

    for folder_name in FOLDERS_TO_PROCESS:
        mod_folder = os.path.join(MOD_SOURCE_DIR, folder_name)
        old_vanilla_folder = os.path.join(OLD_VANILLA_DIR_REFERENCE, folder_name)
        log_message(f"Scanning folder: '{folder_name}'", "INFO", 1)

        if not (os.path.isdir(mod_folder) and os.path.isdir(old_vanilla_folder)):
            log_message(f"Mod or Old Vanilla subfolder for '{folder_name}' not found. Skipping.", "WARN ", 2)
            continue

        for root, _, files in os.walk(mod_folder):
            for filename in files:
                if not filename.lower().endswith(".txt"): continue
                
                mod_rel_path = os.path.relpath(os.path.join(root, filename), MOD_SOURCE_DIR)
                mod_abs_path = os.path.join(MOD_SOURCE_DIR, mod_rel_path)
                old_vanilla_abs_path = os.path.join(OLD_VANILLA_DIR_REFERENCE, mod_rel_path)
                file_level_tweaks = {}

                if not os.path.exists(old_vanilla_abs_path):
                    log_message(f"File '{mod_rel_path}' is in your mod but NOT in OLD VANILLA.", "INFO", 2)
                    mod_added_lines = get_file_lines(mod_abs_path, "reading custom mod file")
                    if mod_added_lines:
                        file_level_tweaks["__FILE_ENTIRELY_ADDED_BY_MOD__"] = "".join(mod_added_lines)
                        log_message(f"Marked '{mod_rel_path}' as a custom file addition.", "DETAIL", 3)
                elif not filecmp.cmp(mod_abs_path, old_vanilla_abs_path, shallow=False):
                    log_message(f"File '{mod_rel_path}' differs from OLD VANILLA. Extracting your changes...", "INFO", 2)
                    mod_lines = get_file_lines(mod_abs_path, "reading mod file for diff")
                    old_vanilla_lines = get_file_lines(old_vanilla_abs_path, "reading old vanilla for diff")

                    if not mod_lines or not old_vanilla_lines:
                        log_message(f"Read error for '{mod_rel_path}' or old vanilla. Skipping diff.", "ERROR", 3)
                        continue

                    mod_line_idx = 0
                    processed_mod_entry_starts = set()
                    while mod_line_idx < len(mod_lines):
                        line = mod_lines[mod_line_idx] # Keep original line for block content
                        match = entry_regex.match(line.strip())
                        entry_processed_this_iter = False
                        if match:
                            entry_name_mod = match.group(1)
                            # Use start index to ensure we only process each unique block once
                            potential_mod_entry_start, _ = find_entry_block_indices(entry_name_mod, mod_lines, mod_line_idx)
                            
                            if potential_mod_entry_start is not None and potential_mod_entry_start not in processed_mod_entry_starts :
                                mod_entry_start, mod_entry_end = potential_mod_entry_start, _ # Re-get end for safety
                                mod_entry_start, mod_entry_end = find_entry_block_indices(entry_name_mod, mod_lines, mod_entry_start) # final boundary

                                if mod_entry_start is not None : # Block found
                                    processed_mod_entry_starts.add(mod_entry_start)
                                    entry_processed_this_iter = True
                                    mod_entry_params = get_parameters_within_block(mod_lines, mod_entry_start, mod_entry_end)
                                    
                                    old_vanilla_entry_start, old_vanilla_entry_end = find_entry_block_indices(entry_name_mod, old_vanilla_lines)
                                    
                                    current_entry_param_tweaks = {}
                                    if old_vanilla_entry_start is not None: # Entry existed in old vanilla
                                        old_van_params = get_parameters_within_block(old_vanilla_lines, old_vanilla_entry_start, old_vanilla_entry_end)
                                        for p_key, (m_val, m_comment) in mod_entry_params.items():
                                            old_val_tuple = old_van_params.get(p_key)
                                            old_val_str = old_val_tuple[0] if old_val_tuple else "__PARAM_ADDED_BY_YOU__"
                                            if p_key not in old_van_params or old_val_str != m_val:
                                                current_entry_param_tweaks[p_key] = (m_val, old_val_str, m_comment)
                                        if not mod_entry_params and old_van_params: # Mod entry is empty but old vanilla had params
                                            log_message(f"Entry '{entry_name_mod}' in mod '{mod_rel_path}' is empty but had params in old vanilla. Marked as cleared by you.", "DETAIL", 3)
                                            # Mark that you cleared it perhaps, or specific logic. For now, this means no tweaks to apply from mod.
                                            # If you want to "remove" params, that's a different logic.
                                            # This script focuses on CHANGING or ADDING params you defined.
                                            pass

                                    else: # Entry is new in your modded file (not in old vanilla file at all)
                                        log_message(f"Entry '{entry_name_mod}' in '{mod_rel_path}' is NEW (not in old vanilla file).", "DETAIL", 3)
                                        entry_block_str = "".join(mod_lines[mod_entry_start : mod_entry_end+1])
                                        file_level_tweaks[f"{entry_name_mod}__ADDED_ENTRY__"] = entry_block_str
                                    
                                    if current_entry_param_tweaks:
                                        file_level_tweaks[entry_name_mod] = current_entry_param_tweaks
                                    mod_line_idx = mod_entry_end # Advance parser
                        if not entry_processed_this_iter:
                            mod_line_idx += 1
                
                if file_level_tweaks:
                    all_user_tweaks[mod_rel_path] = file_level_tweaks
                    files_with_identified_tweaks += 1
    
    log_message(f"Phase 1 Summary: Identified your specific tweaks in {files_with_identified_tweaks} files.", "PHASE")
    return all_user_tweaks

# --- Phase 2: Apply YOUR Extracted Tweaks to NEW VANILLA (with interaction) ---
def apply_tweaks_to_new_vanilla(user_tweaks_map):
    log_message("Phase 2: Applying your tweaks to NEW VANILLA files (interactive)...", "PHASE")
    summary = {
        "files_written": 0, "param_changes_applied": 0, "param_changes_skipped_by_user": 0,
        "entries_added": 0, "entries_skipped_by_user":0, "custom_files_copied": 0, "custom_files_skipped_by_user":0,
        "warn_nv_file_missing": 0, "warn_entry_missing_nv": 0, "warn_param_missing_nv": 0,
        "files_identical_to_nv_skipped": 0
    }
    
    if not user_tweaks_map:
        log_message("No user tweaks from Phase 1. Output mod will be empty.", "INFO", 1)
        return summary

    for mod_rel_path, file_level_tweaks in user_tweaks_map.items():
        log_message(f"Processing file for output: '{mod_rel_path}'", "INFO", 1)
        new_vanilla_abs_path = os.path.join(GAME_VANILLA_DIR_NEW, mod_rel_path)
        output_abs_path = os.path.join(MOD_OUTPUT_DIR, mod_rel_path)
        mod_source_abs_path = os.path.join(MOD_SOURCE_DIR, mod_rel_path) # For filecmp with new_vanilla

        if "__FILE_ENTIRELY_ADDED_BY_MOD__" in file_level_tweaks:
            content_str = file_level_tweaks["__FILE_ENTIRELY_ADDED_BY_MOD__"]
            prompt = f"File '{mod_rel_path}' was entirely created by your mod. Copy to updated mod?"
            user_choice = get_user_confirmation_for_parameter(prompt) # Use same prompter
            if user_choice == "yes":
                log_message(f"Copying your custom-added file '{mod_rel_path}'.", "DETAIL", 2)
                if write_file_lines(output_abs_path, content_str.splitlines(True)):
                    summary["files_written"] += 1; summary["custom_files_copied"] += 1
            else:
                summary["custom_files_skipped_by_user"] +=1
                log_message(f"Skipped custom file '{mod_rel_path}' by user choice.", "INFO", 2)
            continue # Next file

        if not os.path.exists(new_vanilla_abs_path):
            log_message(f"NEW VANILLA file for '{mod_rel_path}' (which you modified) does NOT exist. Your changes cannot be applied.", "WARN ", 2)
            summary["warn_nv_file_missing"] += 1
            continue
        
        if filecmp.cmp(mod_source_abs_path, new_vanilla_abs_path, shallow=False):
            log_message(f"Your modded file '{mod_rel_path}' is ALREADY IDENTICAL to NEW VANILLA. No changes needed, skipping.", "INFO", 2)
            summary["files_identical_to_nv_skipped"] += 1
            continue

        new_vanilla_lines_base = get_file_lines(new_vanilla_abs_path, "reading new vanilla for merge")
        if not new_vanilla_lines_base: continue

        output_lines = list(new_vanilla_lines_base)
        file_actually_changed_by_script = False
        apply_all_for_this_file_mode = False

        for tweak_key, tweak_data in file_level_tweaks.items():
            if tweak_key.endswith("__ADDED_ENTRY__"):
                entry_name_to_add = tweak_key.replace("__ADDED_ENTRY__", "")
                entry_block_to_add_str = tweak_data
                
                prompt_add_entry = f"Entry '{entry_name_to_add}' was added by your mod to '{mod_rel_path}'. Append to NEW vanilla version?"
                user_choice_add_entry = "yes" if apply_all_for_this_file_mode else get_user_confirmation_for_parameter(prompt_add_entry)
                if user_choice_add_entry == "apply_all_for_file": apply_all_for_this_file_mode = True; user_choice_add_entry="yes"
                
                if user_choice_add_entry == "yes":
                    log_message(f"Adding your custom entry '{entry_name_to_add}' to '{mod_rel_path}'.", "DETAIL", 2)
                    if output_lines and not output_lines[-1].endswith(('\n','\r')): output_lines.append('\n')
                    output_lines.append('\n'); output_lines.extend(entry_block_to_add_str.splitlines(True))
                    file_actually_changed_by_script = True; summary["entries_added"] += 1
                    log_message(f"Appended entry '{entry_name_to_add}'. Manual placement review recommended.", "WARN ", 3)
                else:
                    summary["entries_skipped_by_user"] +=1
                    log_message(f"Skipped adding entry '{entry_name_to_add}' by user choice.", "INFO", 3)
            else: # Parameter tweaks for an existing entry
                entry_name_to_update = tweak_key
                your_param_tweaks = tweak_data # {'param': ('mod_val', 'old_van_val', 'mod_comment')}

                nv_entry_start, nv_entry_end = find_entry_block_indices(entry_name_to_update, new_vanilla_lines_base)
                if nv_entry_start is None:
                    log_message(f"Entry '{entry_name_to_update}' you modified in '{mod_rel_path}' NOT found in NEW VANILLA. Tweaks skipped.", "WARN ", 2)
                    summary["warn_entry_missing_nv"] += 1; continue
                
                log_message(f"Checking tweaks for entry '{entry_name_to_update}' in NEW VANILLA '{mod_rel_path}':", "DETAIL", 2)
                
                current_nv_params_in_block = get_parameters_within_block(output_lines, nv_entry_start, nv_entry_end) # Use output_lines as it may have prior changes

                for p_key, (your_mod_val, old_van_val, your_mod_comment) in your_param_tweaks.items():
                    current_nv_val_tuple = current_nv_params_in_block.get(p_key)
                    current_nv_val = current_nv_val_tuple[0] if current_nv_val_tuple else None # Value only
                    
                    if current_nv_val is None: # Param you modded/added doesn't exist in new vanilla entry
                        log_message(f"Param '{p_key}' (YourVal='{your_mod_val}', OldVan='{old_van_val}') NOT in NEW VANILLA entry '{entry_name_to_update}'.", "WARN ", 3)
                        # Potentially prompt to add it - for now, just log as missing.
                        summary["warn_param_missing_nv"] += 1
                        continue

                    if your_mod_val == current_nv_val:
                        log_message(f"Param '{p_key}': YourVal='{your_mod_val}' is SAME as NewVanVal='{current_nv_val}'. No change.", "DEBUG", 3)
                        continue
                    
                    # If we are here, your_mod_val != current_nv_val
                    log_message(f"Potential change for Param '{p_key}' in Entry '{entry_name_to_update}':", "INFO", 3)
                    log_message(f"  Old Vanilla Value: '{old_van_val}'", "INFO", 4)
                    log_message(f"  Your Modded Value: '{your_mod_val}'", "INFO", 4)
                    log_message(f"  New Vanilla Value: '{current_nv_val}'", "INFO", 4)

                    user_choice_param = "yes" if apply_all_for_this_file_mode else get_user_confirmation_for_parameter(f"Apply your value ('{your_mod_val}') for '{p_key}'?")
                    if user_choice_param == "apply_all_for_file": apply_all_for_this_file_mode = True; user_choice_param="yes"

                    if user_choice_param == "yes":
                        applied_successfully = False
                        for line_idx in range(nv_entry_start + 1, nv_entry_end): # Search within original block boundaries
                            # Re-parse the line from output_lines as it might have been changed by a previous param in the same entry
                            iter_nv_line_key, iter_nv_line_val, _, iter_nv_line_comment = parse_parameter_line(output_lines[line_idx])
                            if iter_nv_line_key == p_key:
                                leading_ws = re.match(r"(\s*)", output_lines[line_idx]).group(1)
                                comment_to_add = f" # MODDED: OV='{old_van_val}', NV='{current_nv_val}', Applied='{your_mod_val}'"
                                # Append to existing comment or create new one
                                final_comment = (iter_nv_line_comment + comment_to_add) if iter_nv_line_comment else comment_to_add
                                
                                new_line = f"{leading_ws}{p_key} = {your_mod_val}{final_comment}\n"
                                log_message(f"Applied change for '{p_key}'. New line: {new_line.strip()}", "SUCCESS", 4)
                                output_lines[line_idx] = new_line
                                file_actually_changed_by_script = True
                                summary["param_changes_applied"] += 1
                                applied_successfully = True
                                break
                        if not applied_successfully:
                             log_message(f"Failed to find line for param '{p_key}' again during application. This shouldn't happen.", "ERROR", 4)
                    else:
                        summary["param_changes_skipped_by_user"] += 1
                        log_message(f"Skipped applying tweak for param '{p_key}' by user choice.", "INFO", 4)

        if file_actually_changed_by_script:
            if write_file_lines(output_abs_path, output_lines):
                summary["files_written"] += 1
                log_message(f"Written '{output_abs_path}' to output with changes.", "SUCCESS", 2)
            # else error already logged by write_file_lines
        else:
            log_message(f"No changes ultimately written to '{mod_rel_path}'. File not created in output.", "INFO", 2)

    log_message(f"Phase 2 Summary: Processed application of tweaks.", "PHASE")
    return summary

# --- Main Script Execution ---
def main():
    log_message("Script started. Crusader Kings III Mod Updater (v7 - 3-Way Interactive).", "HEAD")
    global SKIP_ALL_PARAMETER_CONFIRMATIONS
    SKIP_ALL_PARAMETER_CONFIRMATIONS = False
    
    paths_ok = True
    for p_name, p_val in [("MOD_SOURCE_DIR", MOD_SOURCE_DIR), ("OLD_VANILLA_DIR_REFERENCE", OLD_VANILLA_DIR_REFERENCE), ("GAME_VANILLA_DIR_NEW", GAME_VANILLA_DIR_NEW)]:
        if not os.path.isdir(p_val): log_message(f"{p_name} not found: '{p_val}'. Exiting.", "FATAL"); paths_ok = False
    if not paths_ok: return

    log_message(f"Output will be written to: '{MOD_OUTPUT_DIR}'", "INFO")
    log_message("This directory will be DELETED and RECREATED if it exists.", "WARN ")
    
    if input(f"[ACTION ] Proceed with script execution? (yes/no): ").strip().lower() not in ['yes', 'y']:
        log_message("Operation cancelled by user.", "INFO"); return

    if os.path.exists(MOD_OUTPUT_DIR):
        log_message(f"Removing existing output directory '{MOD_OUTPUT_DIR}'...", "DETAIL")
        try: shutil.rmtree(MOD_OUTPUT_DIR)
        except Exception as e: log_message(f"Could not remove output dir: {e}. Exiting.", "FATAL"); return
    try: os.makedirs(MOD_OUTPUT_DIR, exist_ok=True)
    except Exception as e: log_message(f"Could not create output dir: {e}. Exiting.", "FATAL"); return
    log_message(f"Clean output directory prepared: '{MOD_OUTPUT_DIR}'", "SUCCESS")

    user_specific_tweaks = identify_user_tweaks()
    
    if not user_specific_tweaks:
        log_message("No specific user tweaks identified from your mod. Output mod will be empty.", "INFO")
    else:
        final_summary = apply_tweaks_to_new_vanilla(user_specific_tweaks)
        log_message("\n" + ("-" * 30) + " Final Summary (v7) " + ("-" * 30), "HEAD")
        log_message(f"Files written to output mod: {final_summary['files_written']}", "RESULT")
        log_message(f"  Parameter changes applied: {final_summary['param_changes_applied']}", "RESULT_DETAIL")
        log_message(f"  Parameter changes skipped by user: {final_summary['param_changes_skipped_by_user']}", "RESULT_DETAIL")
        log_message(f"  Custom entries added (appended to files): {final_summary['entries_added']}", "RESULT_DETAIL")
        log_message(f"  Custom entries skipped by user: {final_summary['entries_skipped_by_user']}", "RESULT_DETAIL")
        log_message(f"  Custom files (entirely new) copied: {final_summary['custom_files_copied']}", "RESULT_DETAIL")
        log_message(f"  Custom files skipped by user: {final_summary['custom_files_skipped_by_user']}", "RESULT_DETAIL")
        log_message(f"Files skipped (mod already identical to new vanilla): {final_summary['files_identical_to_nv_skipped']}", "RESULT_DETAIL")
        log_message(f"Warnings Encountered:", "RESULT_DETAIL")
        log_message(f"  NEW Vanilla file missing for a modded file: {final_summary['warn_nv_file_missing']}", "RESULT_DETAIL")
        log_message(f"  Entry you modified missing in NEW Vanilla: {final_summary['warn_entry_missing_nv']}", "RESULT_DETAIL")
        log_message(f"  Parameter you modified missing in NEW Vanilla entry: {final_summary['warn_param_missing_nv']}", "RESULT_DETAIL")
        log_message("-" * (78), "HEAD")


    log_message(f"Script finished. Check the '{MOD_OUTPUT_DIR}' directory.", "HEAD")
    log_message("CRITICAL: Manually review all generated files. Use a diff tool!", "WARN ")

if __name__ == "__main__":
    print("--- Crusader Kings III Mod Updater Script (v7 - 3-Way Interactive, In-Line Comments) ---")
    print(f"MOD SOURCE:          '{MOD_SOURCE_DIR}'")
    print(f"OLD VANILLA REF:     '{OLD_VANILLA_DIR_REFERENCE}'")
    print(f"NEW VANILLA GAME:    '{GAME_VANILLA_DIR_NEW}'")
    print(f"OUTPUT TO:           '{MOD_OUTPUT_DIR}'")
    print("-" * 80)
    main()

--- Crusader Kings III Mod Updater Script (v7 - 3-Way Interactive, In-Line Comments) ---
MOD SOURCE:          'C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes'
OLD VANILLA REF:     'C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver'
NEW VANILLA GAME:    'C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game'
OUTPUT TO:           'c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\updated_mod_v7_3way_interactive'
--------------------------------------------------------------------------------
2025-05-25 16:47:28 [HEAD   ] Script started. Crusader Kings III Mod Updater (v7 - 3-Way Interactive).
2025-05-25 16:47:28 [INFO   ] Output will be written to: 'c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\updated_mod_v7_3way_interactive'
2025-05-25 16:47:28 [WARN   ] This directory will be DELETED and RECREATED if it exists.
2025-05-25 16:47:31 [DETAIL ] Removing existing output directory 'c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\updated_mod_v7_3way